# solitonkit demo

This notebook demonstrates the core workflow: generate an
isolated Skyrmion, inspect observables, decompose Baby Skyrme
energy terms, relax the field, save/load it, and export an
animation.

## Visual preview

The static images and GIF below are generated by
`scripts/generate_docs_demo.py`.

![Skyrmion diagnostics](../docs/assets/skyrmion_diagnostics.png)

![Gradient flow](../docs/assets/gradient_flow.gif)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import solitonkit as sk

## 1. Generate a Dirichlet Skyrmion

Dirichlet boundary conditions pin the edge to `n=(0,0,1)`,
which approximates the vacuum at infinity for an isolated
Skyrmion.

In [ ]:
field = sk.make_skyrmion_field(
    96,
    96,
    spacing=0.25,
    radius=5.0,
    charge=1,
    boundary="dirichlet",
)

print("boundary:", field.boundary)
print("topological charge:", sk.topological_charge(field))
print("Baby Skyrme energy:", sk.baby_skyrme_energy(field))

In [ ]:
fig = sk.show_skyrmion_diagnostics(
    field,
    spacing=0.25,
    quiver_step=8,
)

## 2. Decompose the Baby Skyrme energy

In [ ]:
terms = sk.baby_skyrme_energy_terms(
    field,
    kappa=1.0,
    mass=1.0,
    dmi=0.1,
)
terms

![Energy terms](../docs/assets/energy_terms.png)

## 3. Compare relaxation optimizers

In [ ]:
optimizers = [
    ("gradient", sk.run_baby_skyrme_gradient_flow, {"step_size": 1e-4}),
    ("riemannian", sk.run_baby_skyrme_riemannian_gradient_descent, {"step_size": 1e-4}),
    ("barzilai-borwein", sk.run_baby_skyrme_barzilai_borwein, {"initial_step_size": 1e-4}),
    ("lbfgs", sk.run_baby_skyrme_lbfgs, {"initial_step_size": 0.25}),
    ("semi-implicit", sk.run_baby_skyrme_semi_implicit_flow, {"step_size": 1e-3}),
]

for name, method, kwargs in optimizers:
    relaxed, records = method(
        field,
        kappa=0.8,
        mass=0.8,
        steps=25,
        record_every=5,
        **kwargs,
    )
    print(name, "E0 =", records[0].energy, "Efinal =", records[-1].energy)

![Optimizer comparison](../docs/assets/optimizer_energy.png)

## 4. Save, load, and animate

In [ ]:
relaxed, snapshots = sk.run_baby_skyrme_gradient_flow_snapshots(
    field,
    kappa=0.8,
    mass=0.8,
    dmi=0.08,
    step_size=8e-5,
    steps=60,
    frame_every=6,
)

sk.save_field_npz(relaxed, "demo_relaxed.npz")
loaded = sk.load_field_npz("demo_relaxed.npz")

print("loaded boundary:", loaded.boundary)
print("loaded charge:", sk.topological_charge(loaded))

In [ ]:
sk.save_flow_animation(
    snapshots,
    "demo_gradient_flow.gif",
    fps=8,
)

## 5. CLI equivalent

```powershell
solitonkit generate --nx 96 --ny 96 --spacing 0.25 --boundary dirichlet --output field.npz
solitonkit relax --input field.npz --output relaxed.npz --optimizer lbfgs --steps 100
solitonkit plot --input relaxed.npz --output relaxed.png
```